# Validator Judge (LLM 3) — Proof of Concept (v2)

Clean rebuild of the LLM-3 **correctness + uncertainty validator**. Upstream LLMs (1 / 1.2 / 2) are mocked;
the validator is real (Anthropic / `claude-sonnet-4-6`).

Pipeline: **decompose → route → grounded per-claim verdict → worst-case aggregate → selective-prediction UI gate**.

Changes vs v1: within-run N-sample retired (it was inert — temperature gives no diversity on peaked verdicts);
calls parallelized; **reliability handled by a gold-set-tuned threshold (selective prediction), not a
multiplicative calibration**. Design rationale + history: `validator/DECISIONS.md`, `validator/OBSERVATIONS.md`.

## 0. Dependencies

In [1]:
import subprocess, sys
pkgs = ["anthropic", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)
print("Dependencies OK")

Dependencies OK



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 1. Configuration
Anthropic + Sonnet (the verdict stage is entailment reasoning, where a stronger judge helps). Key from a local `.env`.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not API_KEY:
    raise RuntimeError("ANTHROPIC_API_KEY not found. Check your .env file.")

BACKEND         = "anthropic"
ANTHROPIC_MODEL = "claude-sonnet-4-6"   # judge model
OPENAI_MODEL    = "gpt-4o"

print(f"Config  backend={BACKEND}  model={ANTHROPIC_MODEL}  key={'set' if API_KEY else 'MISSING'}")

Config  backend=anthropic  model=claude-sonnet-4-6  key=set


## 2. LLM client + helpers
Thin `LLMClient` wrapper, a JSON-parse helper, and `parallel_map` (LLM calls are I/O-bound → concurrency
gives near-linear speedup within rate limits).

In [3]:
import json, re
from concurrent.futures import ThreadPoolExecutor

class LLMClient:
    """Thin abstraction over Anthropic / OpenAI.  client.complete(system, user) -> str"""
    def __init__(self, backend: str, api_key: str | None = None):
        self.backend = backend.lower()
        if self.backend == "anthropic":
            import anthropic
            key = api_key or os.environ.get("ANTHROPIC_API_KEY")
            self._client = anthropic.Anthropic(api_key=key)
            self._model  = ANTHROPIC_MODEL
        elif self.backend == "openai":
            import openai
            key = api_key or os.environ.get("OPENAI_API_KEY")
            self._client = openai.OpenAI(api_key=key)
            self._model  = OPENAI_MODEL
        else:
            raise ValueError(f"Unknown backend {backend!r}. Choose anthropic | openai")

    def complete(self, system: str, user: str, max_tokens: int = 1024, temperature: float = 0.0) -> str:
        if self.backend == "anthropic":
            msg = self._client.messages.create(
                model=self._model, max_tokens=max_tokens, temperature=temperature,
                system=system, messages=[{"role": "user", "content": user}],
            )
            return msg.content[0].text.strip()
        else:  # openai
            resp = self._client.chat.completions.create(
                model=self._model, max_tokens=max_tokens, temperature=temperature,
                messages=[{"role": "system", "content": system},
                          {"role": "user",   "content": user}],
            )
            return resp.choices[0].message.content.strip()


def parse_json_response(raw: str):
    """Strip ```fences``` and parse JSON."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```[\w]*\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return json.loads(raw)


def parallel_map(fn, items, max_workers=6):
    """Run fn over items concurrently (I/O-bound LLM calls); preserves input order."""
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        return list(ex.map(fn, items))


llm = LLMClient(backend=BACKEND, api_key=API_KEY)
print(f"LLMClient ready  ({BACKEND} / {llm._model})")

LLMClient ready  (anthropic / claude-sonnet-4-6)


## 3. Mock the upstream LLMs — contextualization path (LLM 1 -> LLM 2 -> LLM 3)
Interaction model: the reader selects a passage and clicks a feature button (`define` / `paraphrase` /
`contextualize` / `recall`); the model infers the request. We hard-code that path's output: the grounding
passage (stands in for bounded-retrieval chunks <= reader position, D15), and a **seeded** answer with a
known hallucination (the Jane engagement) so the validator has something to catch.

In [4]:
READER_POSITION = 15   # reader is through chapter 15 of 61; grounding limited to <= this

SELECTED_TEXT = "A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"
FEATURE       = "contextualize"

# Grounding context (stands in for retrieved chunks <= READER_POSITION; curated to isolate the judge, D15).
SOURCE_PASSAGE = """It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.
"My dear Mr. Bennet," said his lady to him one day, "have you heard that Netherfield Park is let at last?"
Mr. Bennet replied that he had not. "But it is," returned she; "for Mrs. Long has just been here, and she told me all about it."
"What is his name?"
"Bingley."
"Is he married or single?"
"Oh! single, my dear, to be sure! A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"
"How so? how can it affect them?"
"My dear Mr. Bennet," replied his wife, "how can you be so tiresome! You must know that I am thinking of his marrying one of them."
"""

# Mocked generator answer, with a seeded error (the Jane engagement).
GENERATED_ANSWER = (
    "Mrs. Bennet is excited because Mr. Bingley is a single man with a large fortune. "
    "She is especially pleased that he is already engaged to her eldest daughter, Jane. "
    "Pride and Prejudice was first published in 1813."
)

print(f"Reader position : through chapter {READER_POSITION} of 61")
print(f"Feature clicked : {FEATURE}")
print(f"Selected text   : {SELECTED_TEXT}\n")
print("Generated answer to validate:\n ", GENERATED_ANSWER)

Reader position : through chapter 15 of 61
Feature clicked : contextualize
Selected text   : A single man of large fortune; four or five thousand a year. What a fine thing for our girls!

Generated answer to validate:
  Mrs. Bennet is excited because Mr. Bingley is a single man with a large fortune. She is especially pleased that he is already engaged to her eldest daughter, Jane. Pride and Prejudice was first published in 1813.


## 4. Stage 4 - Decompose & route
Answer -> atomic claims, each tagged by grounding source (`context` / `world_knowledge`). No verdicts here
(D8). Also splits fact from causal attribution (D7/P2-1); examples are out-of-domain to avoid bias (D17).

In [5]:
SYSTEM_DECOMPOSE = """You are the claim-decomposition stage of a validation system for a reading assistant.

Your job: break an assistant's answer into ATOMIC CLAIMS and tag each by its GROUNDING SOURCE.
You do NOT judge whether claims are true. Extract faithfully: include every claim, even ones that
look wrong, and never correct, soften, or rephrase their meaning.

ATOMIC = one checkable fact per claim. Split conjunctions and compound sentences into separate
claims. Resolve pronouns and references so each claim stands on its own (e.g. "He" -> "the lighthouse keeper").

SEPARATE FACTS FROM ATTITUDES/CAUSES. When a sentence ties a feeling, attitude, belief, or reason to
someone ("X is excited because Y", "X is pleased that Y", "X thinks Y"), split it into:
  (a) the underlying factual claim Y, stated plainly, and
  (b) the attitudinal/causal claim linking the person to it.
Each is checked separately - the fact may be in the passage even when the attribution is not.
Example: "The lighthouse keeper was worried because the lamp had gone out" becomes two claims:
  - "The lamp had gone out."
  - "The lighthouse keeper was worried because the lamp had gone out."

GROUNDING SOURCE: tag each claim as exactly one of
- "context"         : a statement about what happens INSIDE the story (characters, events,
                      relationships, motivations, setting as described in the book). Must be checked
                      against the book passage the reader has read.
- "world_knowledge" : a statement about the real world or the book as a real-world artifact
                      (historical facts, publication dates, real people/places, literary background).
                      Would need an external source to verify.

Output ONLY a JSON array, no prose, no markdown fences. Each element:
  {"claim": "<the atomic claim as a standalone sentence>", "grounding": "context" | "world_knowledge"}
"""

def decompose_and_route(answer: str) -> list[dict]:
    """Answer text -> list of {"claim", "grounding"}. Routing only; no verdicts (D8)."""
    raw = llm.complete(SYSTEM_DECOMPOSE, f"ASSISTANT ANSWER TO DECOMPOSE:\n{answer}", max_tokens=600)
    return parse_json_response(raw)

claims = decompose_and_route(GENERATED_ANSWER)
print(f"{len(claims)} atomic claims:\n")
for i, c in enumerate(claims, 1):
    print(f"{i}. [{c['grounding']:15}] {c['claim']}")

7 atomic claims:

1. [context        ] Mr. Bingley is a single man.
2. [context        ] Mr. Bingley has a large fortune.
3. [context        ] Mrs. Bennet is excited because Mr. Bingley is a single man with a large fortune.
4. [context        ] Mr. Bingley is already engaged to Jane.
5. [context        ] Jane is Mrs. Bennet's eldest daughter.
6. [context        ] Mrs. Bennet is especially pleased that Mr. Bingley is already engaged to Jane.
7. [world_knowledge] Pride and Prejudice was first published in 1813.


## 5. Stage 5 - Grounded per-claim verdict
Each claim gets a verdict (Supported / Partially supported / Contradicted / Unverifiable, D9) grounded
against the passage (D8), plus a verbalized confidence (D5). `world_knowledge` short-circuits to
`Unverifiable` (no web search in this PoC). Single pass - within-run N-sample was retired (it was inert).

In [6]:
from collections import Counter

VERDICT_LABELS = ["Supported", "Partially supported", "Contradicted", "Unverifiable"]

SYSTEM_VERDICT = """You are the per-claim verdict stage of a validation system for a reading assistant.

You are given ONE atomic claim and a SOURCE PASSAGE. Decide how the passage bears on the claim.
Judge ONLY from the passage. Do not use outside knowledge: if the passage does not state or imply the
claim, the verdict is "Unverifiable" - regardless of whether you personally believe the claim is true
or false. Entailment against the provided text - not recall from memory - is the whole point.

Verdict, exactly one of:
- "Supported"           : the passage states or clearly entails the claim.
- "Partially supported" : the passage supports part of the claim but not all of it.
- "Contradicted"        : the passage states or clearly entails the OPPOSITE of the claim.
- "Unverifiable"        : the passage neither supports nor contradicts the claim (it is silent on it).

Tie-break (absence vs. contradiction): use "Contradicted" ONLY when the passage explicitly states or
directly entails the opposite of the claim. If the passage simply does not mention the claim, use
"Unverifiable" - even if other statements in the passage loosely suggest the claim might be false.

Also report your confidence that your verdict is correct, as a number from 0.0 to 1.0.

Output ONLY a JSON object, no prose, no markdown fences:
  {"verdict": "<one label>", "confidence": <0.0-1.0>, "reason": "<one sentence grounded in the passage>"}
"""

def judge_claim_once(claim: str, passage: str, temperature: float = 0.0) -> dict:
    user = f"SOURCE PASSAGE:\n{passage}\n\nCLAIM:\n{claim}"
    raw = llm.complete(SYSTEM_VERDICT, user, max_tokens=300, temperature=temperature)
    return parse_json_response(raw)

def judge_claim(claim: str, grounding: str, passage: str) -> dict:
    """Grounded verdict + verbalized confidence for one claim (single pass)."""
    if grounding == "world_knowledge":
        return {"claim": claim, "grounding": grounding, "verdict": "Unverifiable",
                "verbalized_conf": 1.0, "reason": "world-knowledge claim; no external source in this PoC"}
    v = judge_claim_once(claim, passage, temperature=0.0)
    return {"claim": claim, "grounding": grounding, "verdict": v["verdict"],
            "verbalized_conf": v.get("confidence"), "reason": v.get("reason", "")}

In [7]:
verdicts = parallel_map(lambda c: judge_claim(c["claim"], c["grounding"], SOURCE_PASSAGE), claims)
print(f"Per-claim verdicts ({llm._model}):\n")
print(f"{'#':>2}  {'verdict':<20} {'verb':>5}  claim"); print("-" * 60)
for i, v in enumerate(verdicts, 1):
    vb = f"{v['verbalized_conf']:.2f}" if v["verbalized_conf"] is not None else "  - "
    print(f"{i:>2}  {v['verdict']:<20} {vb:>5}  {v['claim']}")

Per-claim verdicts (claude-sonnet-4-6):

 #  verdict               verb  claim
------------------------------------------------------------
 1  Supported             0.99  Mr. Bingley is a single man.
 2  Supported             0.99  Mr. Bingley has a large fortune.
 3  Supported             0.97  Mrs. Bennet is excited because Mr. Bingley is a single man with a large fortune.
 4  Contradicted          0.95  Mr. Bingley is already engaged to Jane.
 5  Unverifiable          0.95  Jane is Mrs. Bennet's eldest daughter.
 6  Contradicted          0.97  Mrs. Bennet is especially pleased that Mr. Bingley is already engaged to Jane.
 7  Unverifiable          1.00  Pride and Prejudice was first published in 1813.


## 6. Stage 6 - Answer-level aggregation (worst-case)
Any non-`Supported` claim flags the whole answer (D10). Severity: Contradicted > Unverifiable >
Partially supported > Supported.

In [8]:
SEVERITY = {"Supported": 0, "Partially supported": 1, "Unverifiable": 2, "Contradicted": 3}

def aggregate(verdicts: list[dict]) -> dict:
    labels = [v["verdict"] for v in verdicts]
    worst = max(labels, key=lambda l: SEVERITY.get(l, 0))
    flagged = [v for v in verdicts if v["verdict"] == worst and SEVERITY.get(worst, 0) > 0]
    return {"answer_verdict": worst, "counts": Counter(labels), "flagged": flagged, "n_claims": len(labels)}

agg = aggregate(verdicts)
print(f"{agg['n_claims']} claims  |  " + ", ".join(f"{k}x{n}" for k, n in agg["counts"].items()))
print("Answer-level verdict (worst-case):", agg["answer_verdict"])

7 claims  |  Supportedx3, Contradictedx2, Unverifiablex2
Answer-level verdict (worst-case): Contradicted


## 7. Reliability via selective prediction (threshold tuned on a gold set)
We do NOT calibrate confidence with a multiplier. We keep the **raw verbalized confidence as a score** and
**tune the commit/abstain threshold tau on a small gold set** - the selective-prediction / score-based
abstention framing the design doc prescribes (S5 "earn the threshold"; S9 risk-coverage). The judge
**commits** to a verdict when `confidence >= tau` (-> Valid / Not reliable) and **abstains** otherwise
(-> Hedged).

We first check the score is usable at all (**AUROC**: does confidence separate correct from incorrect
verdicts?), then read tau off the **risk-coverage curve** at a target committed-accuracy.

> WARNING: tiny gold set - illustrative only (single annotator, few items). A real threshold needs a large
> hold-out set. See `validator/DECISIONS.md` D18/D19. Refs: selective classification (El-Yaniv & Wiener
> 2010; Geifman & El-Yaniv 2017); score-based abstention for LLM hallucination (Yadkori et al., 2024).

In [9]:
# Gold set (multi-passage). Each item carries its OWN passage. Claims DISJOINT from GENERATED_ANSWER.
# Labels are a SINGLE ANNOTATOR's best judgement -> VERIFY them as the annotator before trusting any number.
# Ch.3 passages are verbatim (3 contiguous excerpts). Illustrative scale only (D18/D19).
PASSAGE_CH1 = SOURCE_PASSAGE   # Ch.1 Bennet/Bingley dialogue

PASSAGE_3_1 = """
Mr. Bingley was good-looking and gentlemanlike; he had a pleasant countenance, and easy, unaffected manners. His sisters were fine women, with an air of decided fashion. His brother-in-law, Mr. Hurst, merely looked the gentleman; but his friend Mr. Darcy soon drew the attention of the room by his fine, tall person, handsome features, noble mien, and the report, which was in general circulation within five minutes after his entrance, of his having ten thousand a year. The gentlemen pronounced him to be a fine figure of a man, the ladies declared he was much handsomer than Mr. Bingley, and he was looked at with great admiration for about half the evening, till his manners gave a disgust which turned the tide of his popularity; for he was discovered to be proud, to be above his company, and above being pleased; and not all his large estate in Derbyshire could then save him from having a most forbidding, disagreeable countenance, and being unworthy to be compared with his friend.
"""

PASSAGE_3_2 = """
Mr. Bingley had soon made himself acquainted with all the principal people in the room; he was lively and unreserved, danced every dance, was angry that the ball closed so early, and talked of giving one himself at Netherfield. Such amiable qualities must speak for themselves. What a contrast between him and his friend! Mr. Darcy danced only once with Mrs. Hurst and once with Miss Bingley, declined being introduced to any other lady, and spent the rest of the evening in walking about the room, speaking occasionally to one of his own party. His character was decided. He was the proudest, most disagreeable man in the world, and every body hoped that he would never come there again. Amongst the most violent against him was Mrs. Bennet, whose dislike of his general behaviour was sharpened into particular resentment, by his having slighted one of her daughters.
"""

PASSAGE_3_3 = """
"Which do you mean?" and turning round, he looked for a moment at Elizabeth, till catching her eye, he withdrew his own and coldly said, "She is tolerable; but not handsome enough to tempt me; and I am in no humour at present to give consequence to young ladies who are slighted by other men. You had better return to your partner and enjoy her smiles, for you are wasting your time with me."
   Mr. Bingley followed his advice. Mr. Darcy walked off; and Elizabeth remained with no very cordial feelings towards him. She told the story, however, with great spirit among her friends; for she had a lively, playful disposition, which delighted in any thing ridiculous.
"""

GOLD_SET = [
    # ---- Ch.1 ----
    {"passage": PASSAGE_CH1, "claim": "Mrs. Long told Mrs. Bennet that Netherfield Park was let.",        "grounding": "context", "gold": "Supported"},
    {"passage": PASSAGE_CH1, "claim": "Mr. Bingley is married.",                                          "grounding": "context", "gold": "Contradicted"},
    {"passage": PASSAGE_CH1, "claim": "Mr. Bingley earns four or five thousand a year.",                  "grounding": "context", "gold": "Supported"},
    {"passage": PASSAGE_CH1, "claim": "Mr. Bingley plans to marry one of the Bennet daughters.",          "grounding": "context", "gold": "Unverifiable"},  # HARD: MRS Bennet's hope, not Bingley's plan
    {"passage": PASSAGE_CH1, "claim": "Mr. Bennet is as eager as his wife about the new neighbour.",      "grounding": "context", "gold": "Contradicted"},   # HARD: he is dry/reluctant
    # ---- Ch.3, passage 3_1 ----
    {"passage": PASSAGE_3_1, "claim": "Mr. Darcy has ten thousand a year.",                               "grounding": "context", "gold": "Supported"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Darcy has a large estate in Derbyshire.",                      "grounding": "context", "gold": "Supported"},
    {"passage": PASSAGE_3_1, "claim": "The ladies thought Mr. Darcy more handsome than Mr. Bingley.",     "grounding": "context", "gold": "Supported"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Hurst is Mr. Bingley's brother-in-law.",                       "grounding": "context", "gold": "Supported"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Darcy made a favourable impression for the whole evening.",    "grounding": "context", "gold": "Contradicted"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Hurst is a wealthy man.",                                      "grounding": "context", "gold": "Unverifiable"},   # HARD: "merely looked the gentleman"
    {"passage": PASSAGE_3_1, "claim": "Mr. Bingley was considered unattractive.",                         "grounding": "context", "gold": "Contradicted"},
    # ---- Ch.3, passage 3_2 ----
    {"passage": PASSAGE_3_2, "claim": "Mr. Bingley danced every dance at the ball.",                      "grounding": "context", "gold": "Supported"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Darcy danced with only two ladies at the ball.",               "grounding": "context", "gold": "Supported"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Bingley wished to host a ball at Netherfield.",                "grounding": "context", "gold": "Supported"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Darcy was well liked by the people at the ball.",              "grounding": "context", "gold": "Contradicted"},
    {"passage": PASSAGE_3_2, "claim": "Mrs. Bennet resented Mr. Darcy.",                                  "grounding": "context", "gold": "Supported"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Bingley thought the ball lasted too long.",                    "grounding": "context", "gold": "Contradicted"},   # HARD: he was angry it closed so EARLY
    {"passage": PASSAGE_3_2, "claim": "Mr. Darcy left the ball early.",                                   "grounding": "context", "gold": "Contradicted"},
    # ---- Ch.3, passage 3_3 ----
    {"passage": PASSAGE_3_3, "claim": "Mr. Darcy thought Elizabeth was very handsome.",                   "grounding": "context", "gold": "Contradicted"},
    {"passage": PASSAGE_3_3, "claim": "Elizabeth told her friends about Mr. Darcy's remark.",             "grounding": "context", "gold": "Supported"},
    {"passage": PASSAGE_3_3, "claim": "Elizabeth was deeply wounded by Mr. Darcy's remark.",              "grounding": "context", "gold": "Contradicted"},   # HARD: she took it playfully
    {"passage": PASSAGE_3_3, "claim": "Elizabeth has a lively, playful disposition.",                     "grounding": "context", "gold": "Supported"},
    {"passage": PASSAGE_3_3, "claim": "Mr. Darcy danced with Elizabeth.",                                 "grounding": "context", "gold": "Contradicted"},
]

def auroc(scores, labels):
    """AUROC of `scores` predicting binary `labels` (1=correct). None if either class is empty."""
    pos = [s for s, y in zip(scores, labels) if y == 1]
    neg = [s for s, y in zip(scores, labels) if y == 0]
    if not pos or not neg:
        return None
    wins = sum((1.0 if sp > sn else 0.5 if sp == sn else 0.0) for sp in pos for sn in neg)
    return wins / (len(pos) * len(neg))

def risk_coverage(scores, correct):
    """For each candidate tau (each distinct score): coverage and committed accuracy."""
    n = len(scores)
    rows = []
    for t in sorted(set(scores)):
        committed = [(s, c) for s, c in zip(scores, correct) if s >= t]
        acc = sum(c for _, c in committed) / len(committed) if committed else None
        rows.append({"tau": t, "coverage": len(committed) / n, "accuracy": acc})
    return rows

def pick_threshold(rows, target_accuracy=0.90, default=0.80):
    """Smallest tau whose committed accuracy >= target (= max coverage among those); fallbacks otherwise."""
    ok = [r for r in rows if r["accuracy"] is not None and r["accuracy"] >= target_accuracy]
    if ok:
        return max(ok, key=lambda r: r["coverage"])["tau"]
    feasible = [r for r in rows if r["accuracy"] is not None]
    return max(feasible, key=lambda r: r["accuracy"])["tau"] if feasible else default

TARGET_ACCURACY = 0.90
gold_preds = parallel_map(lambda g: judge_claim(g["claim"], g["grounding"], g["passage"]), GOLD_SET)
scores  = [p["verbalized_conf"] for p in gold_preds]
correct = [int(p["verdict"] == g["gold"]) for g, p in zip(GOLD_SET, gold_preds)]

gold_accuracy = sum(correct) / len(correct)
score_auroc   = auroc(scores, correct)
rc            = risk_coverage(scores, correct)
CONF_THRESHOLD = pick_threshold(rc, TARGET_ACCURACY)

print(f"Gold set: {len(GOLD_SET)} items | judge accuracy {gold_accuracy:.2f}")
print("AUROC (confidence separates correct vs incorrect): "
      + ("undefined - no errors" if score_auroc is None else f"{score_auroc:.2f}"))
print("\nGold-set detail (MISS = judge != gold; these are what make the score measurable):")
for g, p in zip(GOLD_SET, gold_preds):
    mark = "ok  " if p["verdict"] == g["gold"] else "MISS"
    print(f"  [{mark}] gold={g['gold']:<13} judge={p['verdict']:<19} (verb {p['verbalized_conf']:.2f})  {g['claim']}")
print("\nRisk-coverage curve:")
for r in rc:
    acc = f"{r['accuracy']:.2f}" if r["accuracy"] is not None else "  - "
    print(f"  tau={r['tau']:.2f}  coverage={r['coverage']:.2f}  committed_acc={acc}")
print(f"\nChosen commit threshold tau = {CONF_THRESHOLD:.2f}  (target committed-accuracy >= {TARGET_ACCURACY:.2f})")
if score_auroc is not None:
    print("AUROC is now defined -> we can finally judge whether verbalized confidence is a usable abstention score.")

Gold set: 24 items | judge accuracy 0.92
AUROC (confidence separates correct vs incorrect): 0.95

Gold-set detail (MISS = judge != gold; these are what make the score measurable):
  [ok  ] gold=Supported     judge=Supported           (verb 0.99)  Mrs. Long told Mrs. Bennet that Netherfield Park was let.
  [ok  ] gold=Contradicted  judge=Contradicted        (verb 0.99)  Mr. Bingley is married.
  [ok  ] gold=Supported     judge=Supported           (verb 0.95)  Mr. Bingley earns four or five thousand a year.
  [MISS] gold=Unverifiable  judge=Contradicted        (verb 0.85)  Mr. Bingley plans to marry one of the Bennet daughters.
  [ok  ] gold=Contradicted  judge=Contradicted        (verb 0.92)  Mr. Bennet is as eager as his wife about the new neighbour.
  [ok  ] gold=Supported     judge=Supported           (verb 0.99)  Mr. Darcy has ten thousand a year.
  [ok  ] gold=Supported     judge=Supported           (verb 1.00)  Mr. Darcy has a large estate in Derbyshire.
  [ok  ] gold=Supported   

## 8. Stage 7 - Uncertainty gate + 3-way UI mapping
Gate the answer's driving-claim confidence against the tuned tau (D19): commit (Valid / Not reliable) if
`conf >= tau`, else abstain (Hedged). The high-uncertainty override is checked first.

In [10]:
def map_to_ui(agg: dict, verdicts: list[dict], conf_threshold: float | None = None) -> dict:
    if conf_threshold is None:
        conf_threshold = CONF_THRESHOLD   # tuned on the gold set (D19)
    verdict = agg["answer_verdict"]
    driving = agg["flagged"] if agg["flagged"] else verdicts
    confs = [v["verbalized_conf"] for v in driving if v.get("verbalized_conf") is not None]
    answer_conf = min(confs) if confs else None
    high_uncertainty = (answer_conf is not None) and (answer_conf < conf_threshold)
    if high_uncertainty:            state = "Hedged"
    elif verdict == "Supported":    state = "Valid"
    elif verdict == "Contradicted": state = "Not reliable"
    else:                           state = "Hedged"
    return {"ui_state": state, "answer_verdict": verdict,
            "answer_confidence": answer_conf, "high_uncertainty": high_uncertainty}

BANNER  = {"Valid": "VALID", "Not reliable": "NOT RELIABLE", "Hedged": "HEDGED"}
MESSAGE = {
    "Valid":        "Show the answer with a 'verified' tag, plus reasoning + sources so the reader can check it.",
    "Not reliable": "\"Couldn't give a reliable response - try again?\" (cap retries).",
    "Hedged":       "\"Couldn't fully verify this.\" Show the answer with reasoning + sources exposed.",
}

ui = map_to_ui(agg, verdicts)
conf = f"{ui['answer_confidence']:.2f}" if ui["answer_confidence"] is not None else "n/a"
print(BANNER[ui["ui_state"]])
print(f"  answer verdict    : {ui['answer_verdict']}")
print(f"  answer confidence : {conf}  (commit threshold tau = {CONF_THRESHOLD:.2f})")
print(f"  high uncertainty? : {ui['high_uncertainty']}")
print(f"  UI behaviour      : {MESSAGE[ui['ui_state']]}")

NOT RELIABLE
  answer verdict    : Contradicted
  answer confidence : 0.95  (commit threshold tau = 0.85)
  high uncertainty? : False
  UI behaviour      : "Couldn't give a reliable response - try again?" (cap retries).


## 9. Capstone - the validator as one call
`validate(answer, passage)` chains every stage; `report()` prints it. Swapping the example (or a Phase-2
experiment) is then a one-line call.

In [11]:
def validate(answer: str, passage: str) -> dict:
    claims   = decompose_and_route(answer)
    verdicts = parallel_map(lambda c: judge_claim(c["claim"], c["grounding"], passage), claims)
    agg      = aggregate(verdicts)
    ui       = map_to_ui(agg, verdicts)
    return {"claims": claims, "verdicts": verdicts, "aggregate": agg, "ui": ui}

def report(result: dict) -> None:
    ui, agg = result["ui"], result["aggregate"]
    print(f"{agg['n_claims']} claims  |  " + ", ".join(f"{k}x{n}" for k, n in agg["counts"].items()))
    for i, v in enumerate(result["verdicts"], 1):
        vb = f"{v['verbalized_conf']:.2f}" if v.get("verbalized_conf") is not None else "  - "
        print(f"  {i}. [{v['verdict']:<19}] verb={vb}  {v['claim']}")
    print()
    print(BANNER[ui["ui_state"]], "-", MESSAGE[ui["ui_state"]])
    print(f"  answer verdict {ui['answer_verdict']}  |  high_uncertainty={ui['high_uncertainty']}")

report(validate(GENERATED_ANSWER, SOURCE_PASSAGE))

7 claims  |  Supportedx3, Contradictedx2, Unverifiablex2
  1. [Supported          ] verb=0.99  Mr. Bingley is a single man.
  2. [Supported          ] verb=0.99  Mr. Bingley has a large fortune.
  3. [Supported          ] verb=0.97  Mrs. Bennet is excited because Mr. Bingley is a single man with a large fortune.
  4. [Contradicted       ] verb=0.95  Mr. Bingley is already engaged to Jane.
  5. [Unverifiable       ] verb=0.95  Jane is Mrs. Bennet's eldest daughter.
  6. [Contradicted       ] verb=0.97  Mrs. Bennet is especially pleased that Mr. Bingley is already engaged to Jane.
  7. [Unverifiable       ] verb=1.00  Pride and Prejudice was first published in 1813.

NOT RELIABLE - "Couldn't give a reliable response - try again?" (cap retries).
  answer verdict Contradicted  |  high_uncertainty=False


## 10. Reliability check - run-to-run variability (P2-3)
LLM calls are non-deterministic even at temperature 0, so a single run is not evidence. Run `validate()`
K times on the same input and tally the distribution of the final UI state. This is the offline
reliability metric (S8-S9), parallelized; it never sits in the per-request path.

In [12]:
from collections import Counter as _C
K = 10
# validate() already parallelizes per-claim, so keep the OUTER workers small to respect rate limits.
results = parallel_map(lambda _: validate(GENERATED_ANSWER, SOURCE_PASSAGE), range(K), max_workers=3)
print(f"Ran validate() x{K} (parallel) on the same input:\n")
print("UI state distribution    :", dict(_C(r["ui"]["ui_state"] for r in results)))
print("Claim-count distribution :", dict(_C(r["aggregate"]["n_claims"] for r in results)))
print("Verdict-tuple distribution:")
for tup, n in _C(tuple(v["verdict"] for v in r["verdicts"]) for r in results).most_common():
    print(f"  {n:>2}x  {tup}")

Ran validate() x10 (parallel) on the same input:

UI state distribution    : {'Not reliable': 10}
Claim-count distribution : {7: 10}
Verdict-tuple distribution:
  10x  ('Supported', 'Supported', 'Supported', 'Contradicted', 'Unverifiable', 'Contradicted', 'Unverifiable')
